# Automated Data Quality Monitor

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from jinja2 import Template
from datetime import datetime
import yaml, os, json

## Configuration

In [2]:
config = {
    "dataset": {
        "name": "Healthcare Patient Records",
        "path": "healthcare_sample.csv"
    },
    "thresholds": {
        "max_null_rate":       0.05,   # alert if > 5% nulls per column
        "max_duplicate_rate":  0.02,   # alert if > 2% duplicate rows
        "outlier_method":      "iqr",  # "iqr" or "zscore"
        "iqr_multiplier":      1.5,
        "zscore_threshold":    3.0
    },
    "columns": {
        "numeric": [
            {"name": "age",            "min": 0,   "max": 120},
            {"name": "bmi",            "min": 10,  "max": 80},
            {"name": "blood_pressure", "min": 40,  "max": 250},
            {"name": "cholesterol",    "min": 100, "max": 500},
            {"name": "los_days",       "min": 0,   "max": 365},
            {"name": "total_cost",     "min": 0,   "max": 1_000_000},
        ],
        "categorical": [
            {"name": "gender",    "allowed_values": ["Male", "Female"]},
            {"name": "diagnosis", "allowed_values": ["Diabetes","Hypertension","Healthy","Obesity"]},
        ],
        "required": ["patient_id", "admission_date"]
    }
}

print("✓ Config loaded")


✓ Config loaded


##  Generate Sample Dataset



In [3]:
np.random.seed(42)
n = 500

df_raw = pd.DataFrame({
    "patient_id":     range(1001, 1001 + n),
    "age":            np.random.randint(18, 90, n).astype(float),
    "gender":         np.random.choice(["Male", "Female", "male", "FEMALE", None], n),
    "bmi":            np.round(np.random.normal(27, 5, n), 1),
    "blood_pressure": np.random.randint(60, 180, n).astype(float),
    "cholesterol":    np.random.randint(150, 300, n).astype(float),
    "diagnosis":      np.random.choice(["Diabetes","Hypertension","Healthy","Obesity",None], n),
    "admission_date": pd.date_range("2022-01-01", periods=n, freq="12h").astype(str),
    "los_days":       np.random.randint(1, 30, n).astype(float),
    "total_cost":     np.round(np.random.exponential(3000, n), 2),
})

# Inject intentional quality issues
df_raw.loc[np.random.choice(n, 45, replace=False), "age"]            = np.nan
df_raw.loc[np.random.choice(n, 30, replace=False), "cholesterol"]    = np.nan
df_raw.loc[np.random.choice(n, 10, replace=False), "bmi"]            = 999.0   # outlier
df_raw.loc[np.random.choice(n, 5,  replace=False), "blood_pressure"] = -1.0    # invalid
df_raw.loc[np.random.choice(n, 20, replace=False), "total_cost"]     = np.nan
df_raw = pd.concat([df_raw, df_raw.sample(15, random_state=1)], ignore_index=True)

df_raw.to_csv("healthcare_sample.csv", index=False)

print(f"✓ Dataset generated: {len(df_raw)} rows × {df_raw.shape[1]} columns")
df_raw.head()


✓ Dataset generated: 515 rows × 10 columns


,patient_id,age,gender,bmi,blood_pressure,cholesterol,diagnosis,admission_date,los_days,total_cost
0,1001,69.0,FEMALE,32.5,144.0,NaN,Obesity,2022-01-01 00:00:00,10.0,6295.40
1,1002,32.0,male,27.9,85.0,261.0,None,2022-01-01 12:00:00,4.0,10992.24
2,1003,89.0,None,20.3,131.0,231.0,Hypertension,2022-01-02 00:00:00,8.0,83.71
3,1004,78.0,FEMALE,27.7,107.0,191.0,Healthy,2022-01-02 12:00:00,22.0,5494.30
4,1005,38.0,male,21.7,125.0,170.0,Diabetes,2022-01-03 00:00:00,20.0,NaN


##  Load Dataset

In [4]:
df = pd.read_csv(config["dataset"]["path"])
print(f"✓ Loaded: {len(df)} rows × {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")
df.head()


✓ Loaded: 515 rows × 10 columns
  Columns: ['patient_id', 'age', 'gender', 'bmi', 'blood_pressure', 'cholesterol', 'diagnosis', 'admission_date', 'los_days', 'total_cost']


,patient_id,age,gender,bmi,blood_pressure,cholesterol,diagnosis,admission_date,los_days,total_cost
0,1001,69.0,FEMALE,32.5,144.0,NaN,Obesity,2022-01-01 00:00:00,10.0,6295.40
1,1002,32.0,male,27.9,85.0,261.0,NaN,2022-01-01 12:00:00,4.0,10992.24
2,1003,89.0,NaN,20.3,131.0,231.0,Hypertension,2022-01-02 00:00:00,8.0,83.71
3,1004,78.0,FEMALE,27.7,107.0,191.0,Healthy,2022-01-02 12:00:00,22.0,5494.30
4,1005,38.0,male,21.7,125.0,170.0,Diabetes,2022-01-03 00:00:00,20.0,NaN


## Check 1 — Null / Missing Values

In [5]:
def check_nulls(df, config):
    threshold     = config["thresholds"]["max_null_rate"]
    required_cols = config["columns"].get("required", [])
    results = []
    for col in df.columns:
        null_count  = int(df[col].isna().sum())
        null_rate   = null_count / len(df)
        is_required = col in required_cols
        status = (
            "FAIL"
            if (is_required and null_count > 0) or (null_rate > threshold)
            else "PASS"
        )
        results.append({
            "column":     col,
            "null_count": null_count,
            "null_rate":  round(null_rate * 100, 2),
            "required":   is_required,
            "status":     status,
        })
    total_nulls  = int(df.isna().sum().sum())
    overall_rate = round(total_nulls / df.size * 100, 2)
    return {
        "name":          "Null / Missing Values",
        "overall_rate":  overall_rate,
        "threshold_pct": threshold * 100,
        "details":       results,
        "passed":        sum(1 for r in results if r["status"] == "PASS"),
        "failed":        sum(1 for r in results if r["status"] == "FAIL"),
    }

null_result = check_nulls(df, config)
print(f"Overall null rate : {null_result['overall_rate']}%")
print(f"Passed: {null_result['passed']}  |  Failed: {null_result['failed']}\n")
pd.DataFrame(null_result["details"])


Overall null rate : 6.23%
Passed: 6  |  Failed: 4



,column,null_count,null_rate,required,status
0,patient_id,0,0.00,True,PASS
1,age,48,9.32,False,FAIL
2,gender,108,20.97,False,FAIL
3,bmi,0,0.00,False,PASS
4,blood_pressure,0,0.00,False,PASS
5,cholesterol,30,5.83,False,FAIL
6,diagnosis,115,22.33,False,FAIL
7,admission_date,0,0.00,True,PASS
8,los_days,0,0.00,False,PASS
9,total_cost,20,3.88,False,PASS


## Check 2 — Duplicate Rows


In [6]:
def check_duplicates(df, config):
    threshold = config["thresholds"]["max_duplicate_rate"]
    dup_mask  = df.duplicated(keep="first")
    dup_count = int(dup_mask.sum())
    dup_rate  = dup_count / len(df)
    return {
        "name":            "Duplicate Rows",
        "total_rows":      len(df),
        "duplicate_count": dup_count,
        "duplicate_rate":  round(dup_rate * 100, 2),
        "threshold_pct":   threshold * 100,
        "status":          "FAIL" if dup_rate > threshold else "PASS",
        "sample":          df[dup_mask].head(10).to_dict(orient="records"),
    }

dup_result = check_duplicates(df, config)
print(f"Duplicate rows : {dup_result['duplicate_count']} ({dup_result['duplicate_rate']}%)")
print(f"Status         : {dup_result['status']}\n")
print("Sample duplicate rows:")
pd.DataFrame(dup_result["sample"])


Duplicate rows : 15 (2.91%)
Status         : FAIL

Sample duplicate rows:


,patient_id,age,gender,bmi,blood_pressure,cholesterol,diagnosis,admission_date,los_days,total_cost
0,1305,42.0,FEMALE,24.1,82.0,277.0,NaN,2022-06-02 00:00:00,27.0,4597.26
1,1341,85.0,Female,25.3,80.0,171.0,Obesity,2022-06-20 00:00:00,1.0,4630.36
2,1048,64.0,NaN,25.7,93.0,200.0,Diabetes,2022-01-24 12:00:00,9.0,1491.87
3,1068,57.0,FEMALE,26.4,136.0,254.0,Obesity,2022-02-03 12:00:00,19.0,6791.27
4,1480,34.0,male,30.8,168.0,215.0,Hypertension,2022-08-28 12:00:00,3.0,1466.31
5,1486,NaN,FEMALE,32.8,91.0,277.0,NaN,2022-08-31 12:00:00,14.0,2789.74
6,1311,53.0,FEMALE,36.8,94.0,218.0,Obesity,2022-06-05 00:00:00,8.0,2676.50
7,1032,20.0,NaN,32.1,166.0,157.0,Obesity,2022-01-16 12:00:00,12.0,1359.34
8,1250,47.0,Female,23.6,75.0,296.0,Healthy,2022-05-05 12:00:00,13.0,264.20
9,1091,24.0,Male,26.5,127.0,273.0,NaN,2022-02-15 00:00:00,23.0,3000.32


## Check 3 — Outliers & Range Violations

In [7]:
def check_outliers(df, config):
    method   = config["thresholds"]["outlier_method"]
    iqr_mult = config["thresholds"]["iqr_multiplier"]
    z_thresh = config["thresholds"]["zscore_threshold"]
    results  = []

    for col_cfg in config["columns"].get("numeric", []):
        col    = col_cfg["name"]
        if col not in df.columns:
            continue
        series = df[col].dropna()

        if method == "iqr":
            q1, q3 = series.quantile(0.25), series.quantile(0.75)
            iqr    = q3 - q1
            stat_outliers = int(((series < q1 - iqr_mult * iqr) |
                                  (series > q3 + iqr_mult * iqr)).sum())
        else:
            z = np.abs(stats.zscore(series))
            stat_outliers = int((z > z_thresh).sum())

        col_min = col_cfg.get("min")
        col_max = col_cfg.get("max")
        range_violations = 0
        if col_min is not None: range_violations += int((series < col_min).sum())
        if col_max is not None: range_violations += int((series > col_max).sum())

        results.append({
            "column":           col,
            "method":           method.upper(),
            "stat_outliers":    stat_outliers,
            "range_violations": range_violations,
            "min_allowed":      col_min,
            "max_allowed":      col_max,
            "actual_min":       round(float(series.min()), 2),
            "actual_max":       round(float(series.max()), 2),
            "status":           "FAIL" if (stat_outliers > 0 or range_violations > 0) else "PASS",
        })

    return {
        "name":    "Outliers & Range Violations",
        "method":  method.upper(),
        "details": results,
        "passed":  sum(1 for r in results if r["status"] == "PASS"),
        "failed":  sum(1 for r in results if r["status"] == "FAIL"),
    }

outlier_result = check_outliers(df, config)
print(f"Passed: {outlier_result['passed']}  |  Failed: {outlier_result['failed']}\n")
pd.DataFrame(outlier_result["details"])


Passed: 3  |  Failed: 3



,column,method,stat_outliers,range_violations,min_allowed,max_allowed,actual_min,actual_max,status
0,age,IQR,0,0,0,120,18.00,89.00,PASS
1,bmi,IQR,10,10,10,80,13.10,999.00,FAIL
2,blood_pressure,IQR,0,6,40,250,-1.00,179.00,FAIL
3,cholesterol,IQR,0,0,100,500,150.00,299.00,PASS
4,los_days,IQR,0,0,0,365,1.00,29.00,PASS
5,total_cost,IQR,19,0,0,1000000,6.06,23078.52,FAIL


## Check 4 — Categorical Value Validation

In [8]:
def check_schema(df, config):
    results = []
    for col_cfg in config["columns"].get("categorical", []):
        col     = col_cfg["name"]
        allowed = [v.lower() for v in col_cfg.get("allowed_values", [])]
        if col not in df.columns:
            results.append({"column": col, "status": "MISSING",
                            "invalid_count": 0, "invalid_values": [],
                            "allowed": col_cfg["allowed_values"]})
            continue
        series_clean  = df[col].dropna().astype(str).str.strip()
        invalid_mask  = ~series_clean.str.lower().isin(allowed)
        invalid_vals  = series_clean[invalid_mask].unique().tolist()[:10]
        invalid_count = int(invalid_mask.sum())
        results.append({
            "column":         col,
            "allowed":        col_cfg["allowed_values"],
            "invalid_count":  invalid_count,
            "invalid_values": invalid_vals,
            "status":         "FAIL" if invalid_count > 0 else "PASS",
        })
    return {
        "name":    "Categorical Value Validation",
        "details": results,
        "passed":  sum(1 for r in results if r["status"] == "PASS"),
        "failed":  sum(1 for r in results if r["status"] in ("FAIL","MISSING")),
    }

schema_result = check_schema(df, config)
print(f"Passed: {schema_result['passed']}  |  Failed: {schema_result['failed']}\n")
pd.DataFrame(schema_result["details"])


Passed: 2  |  Failed: 0



,column,allowed,invalid_count,invalid_values,status
0,gender,"[Male, Female]",0,[],PASS
1,diagnosis,"[Diabetes, Hypertension, Healthy, Obesity]",0,[],PASS


## Overall Summary

In [9]:
total_checks = (null_result["passed"]    + null_result["failed"]    +
                outlier_result["passed"] + outlier_result["failed"] +
                schema_result["passed"]  + schema_result["failed"]  + 1)
total_passed = (null_result["passed"]    +
                outlier_result["passed"] +
                schema_result["passed"]  +
                (1 if dup_result["status"] == "PASS" else 0))
total_failed  = total_checks - total_passed
overall_score = round(total_passed / total_checks * 100)

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Overall Score : {overall_score}%")
print(f"  Checks Passed : {total_passed} / {total_checks}")
print(f"  Issues Found  : {total_failed}")
print(f"  Null Rate     : {null_result['overall_rate']}%")
print(f"  Duplicate Rate: {dup_result['duplicate_rate']}%")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Overall Score : 58%
  Checks Passed : 11 / 19
  Issues Found  : 8
  Null Rate     : 6.23%
  Duplicate Rate: 2.91%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## Generate HTML Report

In [10]:
TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Data Quality Report – {{ dataset_name }}</title>
<style>
  :root { --pass:#16a34a; --fail:#dc2626; --warn:#d97706;
          --bg:#f8fafc; --card:#fff; --border:#e2e8f0; --text:#1e293b; --muted:#64748b; }
  * { box-sizing:border-box; margin:0; padding:0; }
  body { font-family:'Segoe UI',system-ui,sans-serif; background:var(--bg);
         color:var(--text); padding:2rem; }
  h1   { font-size:1.6rem; font-weight:700; margin-bottom:.25rem; }
  .meta { color:var(--muted); font-size:.85rem; margin-bottom:2rem; }
  .score-row { display:flex; gap:1rem; flex-wrap:wrap; margin-bottom:2rem; }
  .score-card { flex:1; min-width:150px; background:var(--card);
                border:1px solid var(--border); border-radius:.75rem; padding:1.2rem 1.5rem; }
  .score-card .label { font-size:.75rem; color:var(--muted); text-transform:uppercase;
                        letter-spacing:.05em; margin-bottom:.4rem; }
  .score-card .value { font-size:2rem; font-weight:700; }
  .score-card.pass .value { color:var(--pass); }
  .score-card.fail .value { color:var(--fail); }
  .score-card.neutral .value { color:var(--text); }
  .section { background:var(--card); border:1px solid var(--border);
             border-radius:.75rem; margin-bottom:1.5rem; overflow:hidden; }
  .section-header { display:flex; align-items:center; justify-content:space-between;
                    padding:1rem 1.5rem; border-bottom:1px solid var(--border); }
  .section-header h2 { font-size:1rem; font-weight:600; }
  .badge { display:inline-block; padding:.2rem .7rem; border-radius:9999px;
           font-size:.75rem; font-weight:600; }
  .badge.pass { background:#dcfce7; color:var(--pass); }
  .badge.fail { background:#fee2e2; color:var(--fail); }
  .section-body { padding:1rem 1.5rem; overflow-x:auto; }
  table { width:100%; border-collapse:collapse; font-size:.82rem; }
  th { background:var(--bg); padding:.5rem .75rem; text-align:left; font-weight:600;
       color:var(--muted); font-size:.75rem; text-transform:uppercase; letter-spacing:.04em; }
  td { padding:.5rem .75rem; border-top:1px solid var(--border); }
  tr:hover td { background:#f1f5f9; }
  .pill { display:inline-block; padding:.15rem .5rem; border-radius:9999px;
          font-size:.72rem; font-weight:600; }
  .pill.pass { background:#dcfce7; color:var(--pass); }
  .pill.fail { background:#fee2e2; color:var(--fail); }
  .bar-wrap { display:flex; align-items:center; gap:.5rem; }
  .bar-bg { flex:1; height:6px; background:#e2e8f0; border-radius:3px; }
  .bar-fill { height:6px; border-radius:3px; }
  .bar-fill.ok  { background:var(--pass); }
  .bar-fill.bad { background:var(--fail); }
  footer { text-align:center; color:var(--muted); font-size:.78rem; margin-top:2rem; }
</style>
</head>
<body>
<h1>🔍 Data Quality Report</h1>
<p class="meta">Dataset: <strong>{{ dataset_name }}</strong> &nbsp;·&nbsp;
  Rows: <strong>{{ total_rows }}</strong> &nbsp;·&nbsp;
  Columns: <strong>{{ total_cols }}</strong> &nbsp;·&nbsp;
  Generated: <strong>{{ timestamp }}</strong></p>

<div class="score-row">
  <div class="score-card {{ 'pass' if overall_score >= 80 else 'fail' }}">
    <div class="label">Overall Score</div><div class="value">{{ overall_score }}%</div></div>
  <div class="score-card {{ 'pass' if total_passed == total_checks else 'fail' }}">
    <div class="label">Checks Passed</div>
    <div class="value">{{ total_passed }} / {{ total_checks }}</div></div>
  <div class="score-card {{ 'fail' if total_failed > 0 else 'pass' }}">
    <div class="label">Issues Found</div><div class="value">{{ total_failed }}</div></div>
  <div class="score-card neutral">
    <div class="label">Null Rate</div><div class="value">{{ null_overall }}%</div></div>
  <div class="score-card {{ 'fail' if dup_rate > 0 else 'pass' }}">
    <div class="label">Duplicate Rate</div><div class="value">{{ dup_rate }}%</div></div>
</div>

<div class="section">
  <div class="section-header"><h2>{{ null.name }}</h2>
    <span class="badge {{ 'fail' if null.failed > 0 else 'pass' }}">{{ null.failed }} issue(s)</span></div>
  <div class="section-body">
    <p style="font-size:.82rem;color:var(--muted);margin-bottom:.75rem;">
      Threshold: ≤ {{ null.threshold_pct }}% &nbsp;|&nbsp; Overall: <strong>{{ null.overall_rate }}%</strong></p>
    <table><thead><tr><th>Column</th><th>Null Count</th><th>Null Rate</th>
      <th>Bar</th><th>Required</th><th>Status</th></tr></thead><tbody>
    {% for r in null.details %}<tr>
      <td><strong>{{ r.column }}</strong></td><td>{{ r.null_count }}</td>
      <td>{{ r.null_rate }}%</td>
      <td><div class="bar-wrap"><div class="bar-bg">
        <div class="bar-fill {{ 'bad' if r.status == 'FAIL' else 'ok' }}"
             style="width:{{ [r.null_rate,100]|min }}%"></div></div></div></td>
      <td>{{ '✓' if r.required else '' }}</td>
      <td><span class="pill {{ r.status.lower() }}">{{ r.status }}</span></td>
    </tr>{% endfor %}</tbody></table></div></div>

<div class="section">
  <div class="section-header"><h2>{{ dup.name }}</h2>
    <span class="badge {{ 'fail' if dup.status == 'FAIL' else 'pass' }}">{{ dup.status }}</span></div>
  <div class="section-body">
    <p style="font-size:.82rem;color:var(--muted);">
      Found <strong>{{ dup.duplicate_count }}</strong> duplicates ({{ dup.duplicate_rate }}%
      of {{ dup.total_rows }} rows) &nbsp;·&nbsp; Threshold: ≤ {{ dup.threshold_pct }}%</p>
    {% if dup.sample %}<table style="margin-top:.75rem;"><thead><tr>
      {% for key in dup.sample[0].keys() %}<th>{{ key }}</th>{% endfor %}
    </tr></thead><tbody>
    {% for row in dup.sample %}<tr>{% for v in row.values() %}<td>{{ v }}</td>{% endfor %}</tr>
    {% endfor %}</tbody></table>{% endif %}</div></div>

<div class="section">
  <div class="section-header"><h2>{{ outlier.name }}</h2>
    <span class="badge {{ 'fail' if outlier.failed > 0 else 'pass' }}">{{ outlier.failed }} issue(s)</span></div>
  <div class="section-body">
    <table><thead><tr><th>Column</th><th>Stat. Outliers</th><th>Range Violations</th>
      <th>Allowed Range</th><th>Actual Range</th><th>Status</th></tr></thead><tbody>
    {% for r in outlier.details %}<tr>
      <td><strong>{{ r.column }}</strong></td><td>{{ r.stat_outliers }}</td>
      <td>{{ r.range_violations }}</td>
      <td>{{ r.min_allowed }} – {{ r.max_allowed }}</td>
      <td>{{ r.actual_min }} – {{ r.actual_max }}</td>
      <td><span class="pill {{ r.status.lower() }}">{{ r.status }}</span></td>
    </tr>{% endfor %}</tbody></table></div></div>

<div class="section">
  <div class="section-header"><h2>{{ schema.name }}</h2>
    <span class="badge {{ 'fail' if schema.failed > 0 else 'pass' }}">{{ schema.failed }} issue(s)</span></div>
  <div class="section-body">
    <table><thead><tr><th>Column</th><th>Allowed Values</th>
      <th>Invalid Count</th><th>Invalid Samples</th><th>Status</th></tr></thead><tbody>
    {% for r in schema.details %}<tr>
      <td><strong>{{ r.column }}</strong></td>
      <td>{{ r.allowed | join(', ') }}</td>
      <td>{{ r.invalid_count }}</td>
      <td style="color:var(--fail);font-size:.78rem;">{{ r.invalid_values|join(', ') if r.invalid_values else '—' }}</td>
      <td><span class="pill {{ r.status.lower() }}">{{ r.status }}</span></td>
    </tr>{% endfor %}</tbody></table></div></div>

<footer>Generated by <strong>Automated Data Quality Monitor</strong> &nbsp;·&nbsp;
  Tarane Shanyar &nbsp;·&nbsp; {{ timestamp }}</footer>
</body></html>
"""

from jinja2 import Template

tmpl = Template(TEMPLATE)
html = tmpl.render(
    dataset_name  = config["dataset"]["name"],
    total_rows    = len(df),
    total_cols    = df.shape[1],
    timestamp     = datetime.now().strftime("%Y-%m-%d %H:%M"),
    null          = null_result,
    dup           = dup_result,
    outlier       = outlier_result,
    schema        = schema_result,
    total_checks  = total_checks,
    total_passed  = total_passed,
    total_failed  = total_failed,
    overall_score = overall_score,
    null_overall  = null_result["overall_rate"],
    dup_rate      = dup_result["duplicate_rate"],
)

output_path = f"dq_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)